# Typographic Caustics & Spatial Choreography
## The Poetic Approach: Superposition of Pearcey Beams

This notebook adapts the pseudo-spectral PDE solver to simulate the **spatial choreography** of multiple optical caustics. Instead of a single wave packet, we initialize a precise formation of Pearcey beams—arranged like dancers on a stage or glyphs in a typographic layout—each with distinct spatial offsets and phase shifts.

---

## 1. The Concept: Visualizing Rhythm and Tension

In this simulation, the mathematics of catastrophe theory intersects with spatial choreography and poetics. 
* **The Dancers (Glyphs):** Each Pearcey beam acts as an individual "dancer" or typographic glyph, carrying its own cusp caustic (a sharp, beak-like fold of energy).
* **The Choreography (Meter/Rhyme):** By assigning specific phase offsets ($\phi_k$) and spatial coordinates to each beam, we establish a visual "rhythm."
* **The Interference (Tension/Resolution):** As the formation propagates, the individual cusps do not travel in isolation. Their diffraction fringes overlap and interfere. The sharp, bright lines of the cusps weave through each other, creating a complex, evolving geometric tapestry that visually demonstrates the tension and resolution of the spatial rhythm.

---

## 2. Physical Setup: The Superposition

We initialize $N$ Gaussian wave packets, each propagating in the $+x$ direction with a **cubic phase modulation** in the transverse $y$ direction, shifted by $(x_k, y_k)$ and phase $\phi_k$:

$$
u(x,y,0) = \sum_{k=1}^N \exp\left(-\frac{(x-x_k)^2}{2\sigma_x^2} - \frac{(y-y_k)^2}{2\sigma_y^2}\right) \cos\left(k_x (x-x_k) + \alpha (y-y_k)^3 + \phi_k\right)
$$

---

## 3. The Governing Equation

The medium remains uniform. The governing equation is the standard 2D scalar wave equation, and the principal symbol is simply:

$$
a(\xi, \eta) = c^2 (\xi^2 + \eta^2)
$$

The profound complexity of the resulting visual "glyph" arises entirely from the linear superposition of the initial conditions and the subsequent wave interference!

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed ──
C_SQUARED = 1.0        # c² (m²/s²)
C = np.sqrt(C_SQUARED)

# ── Pearcey Beam Parameters ──
K_X = 12.0             # Longitudinal wavenumber (carrier frequency)
ALPHA_CUBIC = 0.3      # Strength of the cubic phase modulation (cusp sharpness)

# ── Envelope widths ──
SIGMA_X = 2.0          # Longitudinal envelope width
SIGMA_Y = 2.5          # Transverse envelope width

# ── The Choreography (Dancers / Glyphs) ──
# Format: (x_center, y_center, phase_offset)
# Arranged in a dynamic triangular formation with shifting phases (visual rhythm)
DANCERS = [
    (-5.0,  2.5, 0.0),
    ( 0.0, -1.5, np.pi / 3),
    ( 5.0,  2.5, 2 * np.pi / 3)
]

# ── Grid and Time ──
# High resolution is required to resolve the overlapping diffraction fringes
Lx, Ly = 28.0, 14.0
Nx, Ny = 256, 128
Lt, Nt = 12.0, 600
n_frames = 300

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
# We MUST match this convention so the solver receives the correct array orientation.
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t = sp.symbols('x y t', real=True)
xi, eta = sp.symbols('xi eta', real=True)
u_func  = sp.Function('u')
u       = u_func(t, x, y)

# Standard isotropic wave symbol
symbol_wave = C_SQUARED * (xi**2 + eta**2)

print("Principal symbol (Uniform Medium):")
print("  a(ξ, η) =", symbol_wave)

## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u)
#
equation = sp.Eq(
    sp.diff(u, t, 2),
    -psiOp(symbol_wave, u)
)

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp({symbol_wave}, u)")

## 5. Initial conditions (The Choreography)

In [ ]:
def initial_condition_choreography(xx, yy):
    """
    Superposition of multiple Pearcey beams (The Dancers).
    """
    u_total = np.zeros_like(xx)
    
    for (xk, yk, phi_k) in DANCERS:
        # 2D Gaussian envelope centered at (xk, yk)
        env = np.exp(-((xx - xk)**2) / (2 * SIGMA_X**2) - ((yy - yk)**2) / (2 * SIGMA_Y**2))
        
        # Phase: longitudinal carrier + cubic transverse modulation + rhythmic phase offset
        phase = K_X * (xx - xk) + ALPHA_CUBIC * (yy - yk)**3 + phi_k
        
        u_total += env * np.cos(phase)
        
    return u_total

def initial_velocity_choreography(xx, yy):
    """
    Initial velocity based on WKB approximation for rightward propagation: v = -c * du/dx
    We compute the exact spatial derivative for each dancer to prevent initial transients.
    """
    v_total = np.zeros_like(xx)
    
    for (xk, yk, phi_k) in DANCERS:
        env = np.exp(-((xx - xk)**2) / (2 * SIGMA_X**2) - ((yy - yk)**2) / (2 * SIGMA_Y**2))
        phase = K_X * (xx - xk) + ALPHA_CUBIC * (yy - yk)**3 + phi_k
        
        # Derivative of the envelope w.r.t x
        d_env_dx = -((xx - xk) / SIGMA_X**2) * env
        
        # Product rule: d/dx [env * cos(phase)]
        du_dx = d_env_dx * np.cos(phase) - env * K_X * np.sin(phase)
        
        v_total += -C * du_dx
        
    return v_total

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Let the choreography exit the stage cleanly
    initial_condition=initial_condition_choreography,
    initial_velocity=initial_velocity_choreography,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
# Raise the animation size limit
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # Contours are essential to see the woven interference fringes!
    mode='surface',    
    physical=True      
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('typographic_caustics_choreography.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to typographic_caustics_choreography.mp4")